In [1]:
import sys
import pandas as pd
from datetime import datetime, timedelta

# Tell the notebook to look in the main project folder for our code
sys.path.append('..')
from src.data_loader import DataLoader
from src.models import XGBoostPoissonRegressor

# Set pandas display options for cleaner tables
pd.set_option('display.max_columns', None)

In [2]:
# 1. Dynamic 2026 Season Dates
start_date = "2026-03-26"
current_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
end_of_season = "2026-09-27"

print(f"Loading multi-year Statcast data up to {current_date}...")

# 2. Instantiate our Data Loader and pull the stabilized matrix
loader = DataLoader(start_dt=start_date, end_dt=current_date)
df = loader.get_clean_hitter_matrix(min_batted_balls=50)

print(f"\nData loaded successfully! Matrix shape: {df.shape}")

Loading multi-year Statcast data up to 2026-05-27...
Gathering player lookup table. This may take a moment.

--- Processing Historical Lags ---
Fetching Statcast pitches from 2024-03-28 to 2024-05-31...
This is a large query, it may take a moment to complete


c:\Users\mbatt\VSCode\baseball-ros-predictor\venv\Lib\site-packages\pybaseball\statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 65/65 [00:29<00:00,  2.18it/s]


Fetching Statcast pitches from 2025-03-27 to 2025-05-31...
This is a large query, it may take a moment to complete


c:\Users\mbatt\VSCode\baseball-ros-predictor\venv\Lib\site-packages\pybaseball\statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 66/66 [00:22<00:00,  2.90it/s]



--- Processing Current 2026 Context ---
Fetching Statcast pitches from 2026-03-26 to 2026-05-27...
This is a large query, it may take a moment to complete


c:\Users\mbatt\VSCode\baseball-ros-predictor\venv\Lib\site-packages\pybaseball\statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 63/63 [00:30<00:00,  2.10it/s]



Assembling multi-year lag feature matrix...

Data loaded successfully! Matrix shape: (338, 12)


In [3]:
# 1. Define the multi-year feature space
feature_cols = [
    'xwOBA', 'avg_exit_velocity', 'max_exit_velocity', 'avg_launch_angle', 'hard_hit_rate',
    'xwOBA_2025', 'hr_2025', 'xwOBA_2024', 'hr_2024'
]

X = df[feature_cols].values
y = df['home_runs'].values

# 2. Train the Model
print("Training Multi-Year XGBoost Engine...")
model = XGBoostPoissonRegressor()
model.fit(X, y)

# 3. Pacing Calculus
days_played = (datetime.strptime(current_date, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")).days
total_season_days = (datetime.strptime(end_of_season, "%Y-%m-%d") - datetime.strptime(start_date, "%Y-%m-%d")).days
days_remaining = total_season_days - days_played

ros_multiplier = days_remaining / days_played
season_remaining_pct = days_remaining / total_season_days

# 4. Calculate Reliability & Blend
raw_base_projection = model.predict(X)
raw_projected_ros_hrs = raw_base_projection * ros_multiplier

baseline_ros_hrs = df['hr_2025'] * season_remaining_pct
stabilization_constant = 100
reliability_weight = df['total_batted_balls'] / (df['total_batted_balls'] + stabilization_constant)

df['projected_ros_hrs'] = (reliability_weight * raw_projected_ros_hrs) + ((1 - reliability_weight) * baseline_ros_hrs)
df['projected_total_hrs'] = df['home_runs'] + df['projected_ros_hrs']

print("Projections calculated and stabilized!")

Training Multi-Year XGBoost Engine...
Projections calculated and stabilized!


In [ ]:
# Sort by End of Year Total
top_hitters = df.sort_values('projected_total_hrs', ascending=False)

display_cols = [
    'batter_name', 'total_batted_balls', 'max_exit_velocity', 'hard_hit_rate', 
    'home_runs', 'projected_ros_hrs', 'projected_total_hrs'
]

# Display the top 15
top_hitters[display_cols].head(15).style.format({
    'max_exit_velocity': '{:.1f} mph',
    'hard_hit_rate': '{:.1%}',
    'projected_ros_hrs': '+{:.1f}',
    'projected_total_hrs': '{:.1f}'
}).background_gradient(subset=['projected_total_hrs'], cmap='Greens')

,batter_name,total_batted_balls,max_exit_velocity,hard_hit_rate,home_runs,projected_ros_hrs,projected_total_hrs
292,Kyle Schwarber,123,113.2 mph,51.2%,21,+27.9,48.9
489,Yordan Álvarez,162,117.8 mph,53.7%,20,+23.6,43.6
0,Aaron Judge,133,116.2 mph,57.1%,17,+24.9,41.9
350,Munetaka Murakami,118,114.1 mph,59.3%,20,+20.7,40.7
69,Byron Buxton,143,110.4 mph,46.9%,17,+21.2,38.2
36,Ben Rice,134,110.9 mph,53.0%,16,+20.0,36.0
218,James Wood,138,116.3 mph,61.6%,15,+20.9,35.9
325,Matt Olson,161,111.6 mph,51.6%,15,+19.4,34.4
330,Max Muncy,174,111.4 mph,52.3%,14,+17.5,31.5
253,Jordan Walker,148,115.6 mph,52.0%,15,+16.5,31.5
